In [ ]:
import pandas as pd
import yaml
import numpy as np
import re

In [ ]:
import os
os.chdir('../../')

In [ ]:
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

## Import data

### (1) IncoPat only

In [ ]:
IncoPat_only = pd.read_parquet(dataset_config['path_processed'] + 'CN_CN/CN2_IncoPat_only.parquet')
IncoPat_only['source'] = 'IncoPat'
IncoPat_only['year'] = IncoPat_only['year'].astype(pd.Int16Dtype())
IncoPat_only['paperinfo'] = IncoPat_only[['paper_title', 'authors', 'journal_name']].apply(lambda x: list(x), axis=1)
IncoPat_only.drop(columns=['paper_title', 'authors', 'journal_name'], inplace=True)
IncoPat_only['country'] = ('China')
IncoPat_only['apn'] = (
    IncoPat_only['apn']
    .astype('string')
    .str.split('.', n=1).str[0]     # drop .1 etc
    .str.replace(r'^CN', '', regex=True)  # drop leading CN
)
IncoPat_only

### (2) Both

In [ ]:
IncoPat_ROS_both = pd.read_parquet(dataset_config['path_processed'] + 'CN_CN/CN2_IncoPat_OA_intersect.parquet')
IncoPat_ROS_both['source'] = 'Both'
IncoPat_ROS_both.rename(columns={'work_id': 'paperid'}, inplace=True)
IncoPat_ROS_both.drop(columns=['paper_title', 'year'], inplace=True)
IncoPat_ROS_both

### (3) ROS

In [ ]:
ROS_last = pd.read_parquet(dataset_config['path_processed'] + 'CN_CN/POST3_ROS_results.parquet')
ROS_last

### (4) apn and patent_id

In [ ]:
ROS_apn_patentid = pd.read_parquet(dataset_config['path_processed'] + 'CN_CN/CN2_ROS_apn_patentid.parquet')
ROS_apn_patentid

## Merge

In [ ]:
ROS_last = ROS_last.merge(ROS_apn_patentid, on='patent_id')
ROS_last

In [ ]:
ROS_joined = ROS_last.merge(IncoPat_ROS_both, on=['apn', 'paperid'], how='left').drop(columns=['patent_id'])
ROS_joined['source'] = ROS_joined['source'].fillna('ROS')
ROS_joined = ROS_joined.drop_duplicates()
ROS_joined

In [ ]:
ROS_IncoPat_full = pd.concat([ROS_joined, IncoPat_only])[['apn', 'paperid', 'year', 'source', 'paperinfo', 'country']]
ROS_IncoPat_full

In [ ]:
def has_chinese_except_deng(text: str) -> bool:
    """
    Return True only if `text` contains at least one Chinese character
    **other** than the single character '等'.
    """
    s = str(text)
    s = s.replace('等', '')
    return bool(re.search(r'[\u4e00-\u9fff]', s))

In [ ]:
ROS_IncoPat_full['language'] = 'English'

mask_incopat   = ROS_IncoPat_full['source'].eq('IncoPat')
mask_has_zh    = ROS_IncoPat_full['paperinfo'].apply(has_chinese_except_deng)

ROS_IncoPat_full.loc[mask_incopat & mask_has_zh, 'language'] = 'Chinese'

In [ ]:
ROS_IncoPat_full.rename(columns={'year': 'paperyear'}, inplace=True)
ROS_IncoPat_full

In [ ]:
ROS_IncoPat_full.language.value_counts()

In [ ]:
df_incopat = ROS_IncoPat_full[ROS_IncoPat_full['source'] == 'IncoPat']
df_incopat['language'].value_counts(normalize=True) * 100

In [ ]:
ROS_IncoPat_full.source.value_counts(normalize=True) * 100

In [ ]:
ROS_IncoPat_full.to_csv(dataset_config['path_processed'] + 'CN_CN/Chinese_npl_citations_both_sources.csv', index=False)